<a href="https://colab.research.google.com/github/UlfanNayakaDipta/2026-Visi-Komputer/blob/main/JS05_DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q transformers

In [ ]:
import os, time, random, json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CEPAT = DEVICE == "cpu"
print("PyTorch", torch.__version__, "| torchvision", torchvision.__version__)
print("Perangkat:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU saja")

CFG = dict(
    IMG = 128 if CEPAT else 256,
    N_TRAIN = 320 if CEPAT else 2000,
    N_TEST = 100 if CEPAT else 500,
    BATCH = 16,
    EP_UNET = 6 if CEPAT else 15,
    EP_DLV3 = 2 if CEPAT else 6,
    EP_SEGF = 3 if CEPAT else 6,
    BASE_CH = 16 if CEPAT else 32,
)
print("Mode:", "CEPAT (CPU)" if CEPAT else "PENUH (GPU)", CFG)
FIG_DIR = "figures"; os.makedirs(FIG_DIR, exist_ok=True)
WAKTU = {}

## D1: Mengenal Data Segmentasi (Oxford-IIIT Pet)

In [ ]:
import tarfile, urllib.request

URL_IMG = "https://thor.robots.ox.ac.uk/pets/images.tar.gz"
URL_ANN = "https://thor.robots.ox.ac.uk/pets/annotations.tar.gz"

def ambil_subset_pet(root="data/pet_subset", n_train=480, n_test=120):
    os.makedirs(f"{root}/images", exist_ok=True)
    os.makedirs(f"{root}/trimaps", exist_ok=True)
    ann = f"{root}/annotations.tar.gz"
    if not os.path.exists(ann):
        urllib.request.urlretrieve(URL_ANN, ann)
    split, trimaps = {}, {}
    with tarfile.open(ann, "r:gz") as t:
        for m in t:
            if m.name.endswith("annotations/trainval.txt") or m.name.endswith("annotations/test.txt"):
                s = "train" if "trainval" in m.name else "test"
                for line in t.extractfile(m).read().decode().splitlines():
                    if line.strip():
                        split[line.split()[0]] = s
            elif "/trimaps/" in m.name and m.name.endswith(".png") and "._" not in m.name:
                trimaps[os.path.basename(m.name)[:-4]] = t.extractfile(m).read()
    def hitung():
        ada = [f[:-4] for f in os.listdir(f"{root}/images") if f.endswith(".jpg")]
        return (sorted(i for i in ada if split.get(i) == "train"),
                sorted(i for i in ada if split.get(i) == "test"))
    tr, te = hitung()
    if len(tr) >= n_train and len(te) >= n_test:
        print("Subset sudah tersedia di", root)
        return tr[:n_train], te[:n_test]
    with urllib.request.urlopen(URL_IMG) as resp, tarfile.open(fileobj=resp, mode="r|gz") as t:
        for m in t:
            nama = os.path.basename(m.name)
            if not nama.endswith(".jpg"):
                continue
            iid = nama[:-4]; s = split.get(iid)
            if iid not in trimaps or s is None:
                continue
            if (s == "train" and len(tr) >= n_train) or (s == "test" and len(te) >= n_test):
                continue
            with open(f"{root}/images/{nama}", "wb") as f:
                f.write(t.extractfile(m).read())
            with open(f"{root}/trimaps/{iid}.png", "wb") as f:
                f.write(trimaps[iid])
            (tr if s == "train" else te).append(iid)
            if len(tr) >= n_train and len(te) >= n_test:
                break
    tr, te = hitung()
    return tr[:n_train], te[:n_test]

t0 = time.time()
ROOT = "data/pet_subset"
ID_TRAIN, ID_TEST = ambil_subset_pet(ROOT, CFG["N_TRAIN"], CFG["N_TEST"])
print(f"Citra latih: {len(ID_TRAIN)} | citra uji: {len(ID_TEST)} | waktu {time.time()-t0:.1f} detik")
print("Jumlah ras pada data latih:", len({i.rsplit('_', 1)[0] for i in ID_TRAIN}))

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def muat_pasangan(iid, size):
    img = Image.open(f"{ROOT}/images/{iid}.jpg").convert("RGB").resize((size, size), Image.BILINEAR)
    tri = Image.open(f"{ROOT}/trimaps/{iid}.png").resize((size, size), Image.NEAREST)
    return np.array(img), np.array(tri)

class PetSegDataset(torch.utils.data.Dataset):
    def __init__(self, ids, size, augment=False):
        pairs = [muat_pasangan(i, size) for i in ids]
        self.img = torch.from_numpy(np.stack([p[0] for p in pairs])).permute(0, 3, 1, 2)
        self.tri = torch.from_numpy(np.stack([p[1] for p in pairs]))
        self.augment = augment
    def __len__(self):
        return len(self.img)
    def __getitem__(self, i):
        x = self.img[i].float() / 255.0
        tri = self.tri[i]
        y = (tri == 1).float().unsqueeze(0)
        valid = (tri != 3).float().unsqueeze(0)
        if self.augment and random.random() < 0.5:
            x, y, valid = x.flip(-1), y.flip(-1), valid.flip(-1)
        return (x - MEAN) / STD, y, valid

t0 = time.time()
ds_train = PetSegDataset(ID_TRAIN, CFG["IMG"], augment=True)
ds_test = PetSegDataset(ID_TEST, CFG["IMG"], augment=False)
dl_train = torch.utils.data.DataLoader(ds_train, batch_size=CFG["BATCH"], shuffle=True)
dl_test = torch.utils.data.DataLoader(ds_test, batch_size=CFG["BATCH"], shuffle=False)
x, y, v = ds_train[0]
print("Bentuk x:", tuple(x.shape), "| y:", tuple(y.shape), "| valid:", tuple(v.shape))
tri_all = ds_train.tri
print("Proporsi piksel hewan: %.3f | latar: %.3f | batas: %.3f" % tuple(((tri_all == k).float().mean().item()) for k in (1, 2, 3)))
WAKTU["D1 data"] = time.time() - t0

In [ ]:
def denorm(x):
    return (x * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

fig, ax = plt.subplots(3, 4, figsize=(12, 9))
for k in range(4):
    x, y, v = ds_test[k]
    ax[0, k].imshow(denorm(x)); ax[0, k].set_title(ID_TEST[k], fontsize=9)
    ax[1, k].imshow(ds_test.tri[k], cmap="viridis", vmin=1, vmax=3); ax[1, k].set_title("trimap (1 hewan, 2 latar, 3 batas)", fontsize=8)
    ax[2, k].imshow(y[0], cmap="gray"); ax[2, k].set_title("masker biner", fontsize=9)
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/js05_d1_contoh_data.png", dpi=120); plt.show()

## D2: Membangun dan Melatih U-Net dari Nol

In [ ]:
class DoubleConv(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=16):
        super().__init__()
        c = [base, base*2, base*4, base*8, base*16]
        self.enc = nn.ModuleList([DoubleConv(in_ch, c[0])] + [DoubleConv(c[i], c[i+1]) for i in range(4)])
        self.up = nn.ModuleList([nn.ConvTranspose2d(c[i+1], c[i], 2, stride=2) for i in reversed(range(4))])
        self.dec = nn.ModuleList([DoubleConv(c[i]*2, c[i]) for i in reversed(range(4))])
        self.head = nn.Conv2d(c[0], out_ch, 1)
    def forward(self, x):
        skips = []
        for i, blk in enumerate(self.enc):
            x = blk(x if i == 0 else F.max_pool2d(x, 2))
            skips.append(x)
        x = skips.pop()
        for up, dec in zip(self.up, self.dec):
            x = up(x)
            x = dec(torch.cat([x, skips.pop()], dim=1))
        return self.head(x)

unet = UNet(base=CFG["BASE_CH"]).to(DEVICE)
print("Jumlah parameter U-Net: %.2f juta" % (sum(p.numel() for p in unet.parameters()) / 1e6))
print("Uji bentuk keluaran:", tuple(unet(torch.zeros(2, 3, CFG["IMG"], CFG["IMG"], device=DEVICE)).shape))

In [ ]:
def bce_dice_loss(logit, y, valid, w_dice=1.0, eps=1.0):
    bce = F.binary_cross_entropy_with_logits(logit, y, reduction="none")
    bce = (bce * valid).sum() / valid.sum().clamp_min(1)
    p = torch.sigmoid(logit) * valid
    inter = (p * y * valid).sum(dim=(1, 2, 3))
    dice = 1 - (2 * inter + eps) / (p.sum(dim=(1, 2, 3)) + (y * valid).sum(dim=(1, 2, 3)) + eps)
    return bce + w_dice * dice.mean()

def latih(model, forward, epochs, lr, nama, param_groups=None):
    opt = torch.optim.AdamW(param_groups or model.parameters(), lr=lr, weight_decay=1e-4)
    riwayat = {"train": [], "test": []}
    t0 = time.time()
    for ep in range(epochs):
        model.train(); tot = 0
        for x, y, v in dl_train:
            x, y, v = x.to(DEVICE), y.to(DEVICE), v.to(DEVICE)
            loss = bce_dice_loss(forward(model, x), y, v)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(x)
        model.eval(); tot_te = 0
        with torch.no_grad():
            for x, y, v in dl_test:
                x, y, v = x.to(DEVICE), y.to(DEVICE), v.to(DEVICE)
                tot_te += bce_dice_loss(forward(model, x), y, v).item() * len(x)
        riwayat["train"].append(tot / len(ds_train)); riwayat["test"].append(tot_te / len(ds_test))
        print(f"[{nama}] epoch {ep+1}/{epochs} loss latih {riwayat['train'][-1]:.4f} loss uji {riwayat['test'][-1]:.4f} ({time.time()-t0:.0f} s)")
    WAKTU[nama] = time.time() - t0
    return riwayat

fwd_unet = lambda m, x: m(x)
set_seed(42)
hist_unet = latih(unet, fwd_unet, CFG["EP_UNET"], 1e-3, "U-Net (dari nol)")

In [ ]:
plt.figure(figsize=(6, 4))
ep = range(1, len(hist_unet["train"]) + 1)
plt.plot(ep, hist_unet["train"], "o-", label="Loss latih")
plt.plot(ep, hist_unet["test"], "s-", label="Loss uji")
plt.xlabel("Epoch"); plt.ylabel("BCE + Dice"); plt.title("Kurva loss U-Net dari nol"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/js05_d2_kurva_unet.png", dpi=120); plt.show()

## D3: Metrik Evaluasi Segmentasi

In [ ]:
@torch.no_grad()
def evaluasi(model, forward, thr=0.5):
    model.eval()
    TP = FP = FN = TN = 0; iou_per_citra = []
    for x, y, v in dl_test:
        x, y, v = x.to(DEVICE), y.to(DEVICE), v.to(DEVICE)
        pred = (torch.sigmoid(forward(model, x)) > thr).float()
        tp = (pred * y * v).sum(dim=(1, 2, 3)); fp = (pred * (1 - y) * v).sum(dim=(1, 2, 3))
        fn = ((1 - pred) * y * v).sum(dim=(1, 2, 3)); tn = ((1 - pred) * (1 - y) * v).sum(dim=(1, 2, 3))
        TP += tp.sum().item(); FP += fp.sum().item(); FN += fn.sum().item(); TN += tn.sum().item()
        iou_per_citra += (tp / (tp + fp + fn).clamp_min(1)).tolist()
    iou_fg = TP / (TP + FP + FN); iou_bg = TN / (TN + FN + FP)
    return {"IoU_hewan": iou_fg, "IoU_latar": iou_bg, "mIoU": (iou_fg + iou_bg) / 2,
            "Dice": 2 * TP / (2 * TP + FP + FN), "PixelAcc": (TP + TN) / (TP + TN + FP + FN),
            "iou_per_citra": iou_per_citra}

def ringkas(m):
    return {k: round(v, 4) for k, v in m.items() if k != "iou_per_citra"}

HASIL = {}
HASIL["U-Net (dari nol)"] = evaluasi(unet, fwd_unet)
print(ringkas(HASIL["U-Net (dari nol)"]))

In [ ]:
@torch.no_grad()
def prediksi(model, forward, idx):
    x = torch.stack([ds_test[i][0] for i in idx]).to(DEVICE)
    return (torch.sigmoid(forward(model, x)) > 0.5).float().cpu()

idx_vis = [0, 1, 2, 3, 4]
p_unet = prediksi(unet, fwd_unet, idx_vis)
fig, ax = plt.subplots(3, len(idx_vis), figsize=(3 * len(idx_vis), 9))
for j, i in enumerate(idx_vis):
    x, y, v = ds_test[i]
    ax[0, j].imshow(denorm(x)); ax[0, j].set_title("citra", fontsize=9)
    ax[1, j].imshow(y[0], cmap="gray"); ax[1, j].set_title("ground truth", fontsize=9)
    ax[2, j].imshow(p_unet[j, 0], cmap="gray"); ax[2, j].set_title("prediksi U-Net", fontsize=9)
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/js05_d3_prediksi_unet.png", dpi=120); plt.show()

## D4: Transfer Learning dengan DeepLabV3

In [ ]:
from torchvision.models.segmentation import deeplabv3_mobilenet_v3_large, DeepLabV3_MobileNet_V3_Large_Weights

set_seed(42)
dlv3 = deeplabv3_mobilenet_v3_large(weights=DeepLabV3_MobileNet_V3_Large_Weights.DEFAULT)
dlv3.classifier[4] = nn.Conv2d(256, 1, kernel_size=1)
dlv3.aux_classifier = None
dlv3 = dlv3.to(DEVICE)
print("Jumlah parameter DeepLabV3-MobileNetV3: %.2f juta" % (sum(p.numel() for p in dlv3.parameters()) / 1e6))

fwd_dlv3 = lambda m, x: m(x)["out"]
groups = [{"params": dlv3.backbone.parameters(), "lr": 1e-4},
          {"params": dlv3.classifier.parameters(), "lr": 1e-3}]
hist_dlv3 = latih(dlv3, fwd_dlv3, CFG["EP_DLV3"], 1e-3, "DeepLabV3-MobileNetV3 (pra-latih)", param_groups=groups)
HASIL["DeepLabV3-MobileNetV3 (pra-latih)"] = evaluasi(dlv3, fwd_dlv3)
print(ringkas(HASIL["DeepLabV3-MobileNetV3 (pra-latih)"]))

## D5: Fine-Tuning SegFormer-B0

In [ ]:
import transformers
from transformers import SegformerForSemanticSegmentation
transformers.logging.set_verbosity_error()

set_seed(42)
segf = SegformerForSemanticSegmentation.from_pretrained("nvidia/mit-b0", num_labels=1).to(DEVICE)
print("Jumlah parameter SegFormer-B0: %.2f juta" % (sum(p.numel() for p in segf.parameters()) / 1e6))

def fwd_segf(m, x):
    logit = m(pixel_values=x).logits
    return F.interpolate(logit, size=x.shape[-2:], mode="bilinear", align_corners=False)

hist_segf = latih(segf, fwd_segf, CFG["EP_SEGF"], 3e-4, "SegFormer-B0 (pra-latih)")
HASIL["SegFormer-B0 (pra-latih)"] = evaluasi(segf, fwd_segf)
print(ringkas(HASIL["SegFormer-B0 (pra-latih)"]))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, (nama, h) in zip(ax, [("DeepLabV3-MobileNetV3", hist_dlv3), ("SegFormer-B0", hist_segf)]):
    ep = range(1, len(h["train"]) + 1)
    a.plot(ep, h["train"], "o-", label="Loss latih"); a.plot(ep, h["test"], "s-", label="Loss uji")
    a.set_title(nama); a.set_xlabel("Epoch"); a.set_ylabel("BCE + Dice"); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/js05_d5_kurva_pralatih.png", dpi=120); plt.show()

## D6: Perbandingan Model dan Analisis Kesalahan

In [ ]:
import pandas as pd

param = {"U-Net (dari nol)": unet, "DeepLabV3-MobileNetV3 (pra-latih)": dlv3, "SegFormer-B0 (pra-latih)": segf}
tabel = pd.DataFrame([{"Model": k, **ringkas(v), "Parameter (juta)": round(sum(p.numel() for p in param[k].parameters()) / 1e6, 2),
                       "Waktu latih (s)": round(WAKTU[k], 1)} for k, v in HASIL.items()])
print(tabel.to_string(index=False))
tabel.to_csv(f"{FIG_DIR}/js05_tabel_hasil.csv", index=False)
with open(f"{FIG_DIR}/js05_hasil.json", "w") as f:
    json.dump({"perangkat": DEVICE, "mode": "CEPAT" if CEPAT else "PENUH", "cfg": CFG,
               "hasil": {k: ringkas(v) for k, v in HASIL.items()}, "waktu_detik": WAKTU}, f, indent=2)

In [ ]:
model_fwd = [("U-Net", unet, fwd_unet), ("DeepLabV3", dlv3, fwd_dlv3), ("SegFormer-B0", segf, fwd_segf)]
idx_vis = [0, 5, 10, 15]
fig, ax = plt.subplots(len(idx_vis), 5, figsize=(15, 3 * len(idx_vis)))
preds = {n: prediksi(m, f, idx_vis) for n, m, f in model_fwd}
for r, i in enumerate(idx_vis):
    x, y, v = ds_test[i]
    ax[r, 0].imshow(denorm(x)); ax[r, 1].imshow(y[0], cmap="gray")
    ax[r, 0].set_title("citra"); ax[r, 1].set_title("ground truth")
    for c, (n, _, _) in enumerate(model_fwd):
        ax[r, c + 2].imshow(preds[n][r, 0], cmap="gray"); ax[r, c + 2].set_title(n)
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/js05_d6_perbandingan.png", dpi=110); plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
for n, m in HASIL.items():
    plt.hist(m["iou_per_citra"], bins=20, range=(0, 1), alpha=.5, label=n)
plt.xlabel("IoU per citra (kelas hewan)"); plt.ylabel("Jumlah citra"); plt.legend(fontsize=8); plt.grid(alpha=.3)
plt.title("Distribusi IoU per citra pada data uji")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/js05_d6_histogram_iou.png", dpi=120); plt.show()

terbaik = max(HASIL, key=lambda k: HASIL[k]["mIoU"])
mdl, fw = {n: (m, f) for n, m, f in [("U-Net (dari nol)", unet, fwd_unet), ("DeepLabV3-MobileNetV3 (pra-latih)", dlv3, fwd_dlv3), ("SegFormer-B0 (pra-latih)", segf, fwd_segf)]}[terbaik]
iou = np.array(HASIL[terbaik]["iou_per_citra"]); buruk = np.argsort(iou)[:4].tolist()
pb = prediksi(mdl, fw, buruk)
fig, ax = plt.subplots(2, 4, figsize=(14, 7))
for j, i in enumerate(buruk):
    x, y, v = ds_test[i]; p = pb[j, 0]; g = y[0]; val = v[0]
    peta = np.zeros((*g.shape, 3))
    peta[((p == 1) & (g == 1) & (val == 1)).numpy()] = [0, .8, 0]
    peta[((p == 1) & (g == 0) & (val == 1)).numpy()] = [1, 0, 0]
    peta[((p == 0) & (g == 1) & (val == 1)).numpy()] = [0, .3, 1]
    ax[0, j].imshow(denorm(x)); ax[0, j].set_title(f"{ID_TEST[i]}\nIoU = {iou[i]:.2f}", fontsize=9)
    ax[1, j].imshow(peta); ax[1, j].set_title("hijau TP, merah FP, biru FN", fontsize=9)
for a in ax.ravel(): a.axis("off")
plt.suptitle(f"Empat kasus terburuk model {terbaik}")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/js05_d6_kasus_terburuk.png", dpi=110); plt.show()
print("Model terbaik:", terbaik, "| IoU terendah:", np.round(iou[buruk], 3).tolist())

## Eksperimen Perbaikan Model Terbaik (Menambah Epoch SegFormer-B0)

In [ ]:
set_seed(42)
segf_exp = SegformerForSemanticSegmentation.from_pretrained("nvidia/mit-b0", num_labels=1).to(DEVICE)
ep_tambahan = CFG["EP_SEGF"] + 4
hist_segf_exp = latih(segf_exp, fwd_segf, ep_tambahan, 3e-4, f"SegFormer-B0 ({ep_tambahan} epoch)")
hasil_exp = evaluasi(segf_exp, fwd_segf)

print("Perbandingan Eksperimen Perbaikan:")
print("Baseline SegFormer-B0 :", ringkas(HASIL["SegFormer-B0 (pra-latih)"]))
print(f"Eksperimen ({ep_tambahan} epoch):", ringkas(hasil_exp))

## Segmentasi Minimal 3 Foto Hewan Sendiri + Tampilan Overlay

In [ ]:
from google.colab import files as colab_files

uploaded_pets = colab_files.upload()
nama_files = list(uploaded_pets.keys())

fig, ax = plt.subplots(len(nama_files), 3, figsize=(12, 4 * len(nama_files)))
if len(nama_files) == 1:
    ax = np.expand_dims(ax, axis=0)

mdl_best, fw_best = segf_exp, fwd_segf
mdl_best.eval()

for idx, fn in enumerate(nama_files):
    img_pil = Image.open(fn).convert("RGB").resize((CFG["IMG"], CFG["IMG"]), Image.BILINEAR)
    img_np = np.array(img_pil) / 255.0
    x_in = ((torch.from_numpy(img_np).permute(2, 0, 1).float() - MEAN) / STD).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        mask_pred = (torch.sigmoid(fw_best(mdl_best, x_in)) > 0.5).float().cpu()[0, 0].numpy()

    overlay = img_np.copy()
    overlay[mask_pred == 1] = overlay[mask_pred == 1] * 0.5 + np.array([0.0, 1.0, 0.0]) * 0.5

    ax[idx, 0].imshow(img_np); ax[idx, 0].set_title(f"Citra Asli: {fn}")
    ax[idx, 1].imshow(mask_pred, cmap="gray"); ax[idx, 1].set_title("Prediksi Masker Biner")
    ax[idx, 2].imshow(overlay); ax[idx, 2].set_title("Hasil Overlay Segmentasi")

for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/js05_penugasan_foto_sendiri.png", dpi=120); plt.show()